# 01 - State, State Schema, State Reducer, Multiple Schemas

Basics of LangGraph state: how it's shaped, how updates combine, and how a graph
can accept a different shape than it returns.

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

load_dotenv()

## State Schema

State is just a typed dict describing the fields that flow through the graph.

In [ ]:
State = TypedDict("State", {"count": int, "message": str})

def increment(state):
    new_count = state["count"] + 1
    return {"count": new_count, "message": f"count is now {new_count}"}

graph = StateGraph(State)
graph.add_node("increment", increment)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)
app = graph.compile()

app.invoke({"count": 0, "message": ""})

## State Reducer

By default a node's return value overwrites a field. A reducer (e.g. `operator.add`)
tells LangGraph how to *combine* the old value with the new one instead.

In [ ]:
ReducerState = TypedDict("ReducerState", {"nums": Annotated[list, operator.add]})

def add_one(state):
    return {"nums": [1]}

def add_two(state):
    return {"nums": [2]}

graph2 = StateGraph(ReducerState)
graph2.add_node("add_one", add_one)
graph2.add_node("add_two", add_two)
graph2.add_edge(START, "add_one")
graph2.add_edge("add_one", "add_two")
graph2.add_edge("add_two", END)
app2 = graph2.compile()

app2.invoke({"nums": []})  # -> {'nums': [1, 2]}, appended instead of overwritten

## Multiple Schemas

A graph's internal state can be wider than what callers pass in or see back.
Give `StateGraph` separate `input=` / `output=` schemas to filter both ends.

In [ ]:
InputState = TypedDict("InputState", {"question": str})
OutputState = TypedDict("OutputState", {"answer": str})
OverallState = TypedDict("OverallState", {"question": str, "answer": str, "scratch": str})

def answer_node(state):
    return {"answer": f"Answer to: {state['question']}", "scratch": "internal notes"}

graph3 = StateGraph(OverallState, input=InputState, output=OutputState)
graph3.add_node("answer_node", answer_node)
graph3.add_edge(START, "answer_node")
graph3.add_edge("answer_node", END)
app3 = graph3.compile()

app3.invoke({"question": "What is LangGraph?"})  # only 'answer' comes back, 'scratch' is filtered out